In [ ]:
%pip install numpy
%pip install matplotlib
%pip install tqdm
%pip install gymnasium
%pip install torch
%pip install tensorboard

## Environment

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from enum import Enum
import gymnasium as gym
import tqdm

MAP = [
    "SFFF",
    "FHFH",
    "FFFH",
    "HFFG"
]

GAMMA = 0.999

def create_environment(success_rate):
  return gym.make("FrozenLake-v1", map_name="4x4", render_mode='rgb_array', success_rate=success_rate, is_slippery=True, desc=MAP)

env = create_environment(0.5)
env.reset()
env.render()

## Utilities

In [ ]:
LEFT = 0
DOWN = 1
RIGHT = 2
UP = 3

def get_free_cells(map_str):
  """Returns the number of cells that the agent can be in.
  """
  free_cells = []
  for i, row in enumerate(map_str):
    for j, cell in enumerate(row):
      if cell in 'SF':
        free_cells.append(4 * i + j)
  return free_cells

def visualize_policy(ax, map_str, V, pi):
  """Displays the value function and policy as a grid.
  """

  H = len(map_str)
  W = len(map_str[0])

  free_cells = get_free_cells(map_str)
  V_viz = V.copy()
  V_viz[:] = None
  V_viz[free_cells] = V[free_cells]
  h = ax.imshow(V_viz.reshape([H, W]), vmin=0.0, vmax=1.0)

  u = (pi == RIGHT).astype(float) - (pi == LEFT).astype(float)
  v = (pi == UP).astype(float) - (pi == DOWN).astype(float)
  uu = float('nan') * u
  uu[free_cells] = u[free_cells]
  u = uu

  u = u.reshape([H, W])
  v = v.reshape([H, W])

  ax.quiver(
      list(range(W)),
      list(range(H)),
      u,
      v,
      color='r',
  )

  return h

## Temporal Difference Policy Iteration

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
from torch.utils.tensorboard import SummaryWriter
import datetime

def get_greedy_policy_from_Q(Q):
  return np.argmax(Q, axis=-1)

def eval_policy_td(env, map_str, pi, Q0, discount_factor, learning_rate, num_steps):

  def _reset_environment():
    env.reset()
    state = np.random.choice(free_cells)
    env.unwrapped.s = state
    action = env.action_space.sample()
    return state, action

  free_cells = get_free_cells(map_str)
  Q = Q0.copy()

  state, action = _reset_environment()

  #############################################################
  # We now update Q-values after each step. No need for the
  # episode outerloop we had in MC.
  #############################################################

  for t in range(num_steps):
    next_state, reward, done, truncated, _ = env.step(action)

    target = reward
    if not done:
      target += discount_factor * Q[next_state, pi[next_state]]

    Q[state, action] += learning_rate * (target - Q[state, action])

    if done or truncated:
      state, action = _reset_environment()
    else:
      state = next_state
      action = pi[state]

  return Q, {}

def policy_average_return(env, pi, discount_factor, num_episodes):
  G = 0

  for i in range(num_episodes):
    state = env.reset()[0]
    gamma_t = 1.0

    done = False
    while not done:
      action = pi[state]
      state, reward, done, truncated, _ = env.step(action)
      done = done or truncated

      G += gamma_t * reward
      gamma_t *= discount_factor

  return G / num_episodes

NUM_EPOCHS = 500
NUM_STEPS = 10
LOG_RATE = 1

env = create_environment(0.5)
free_cells = get_free_cells(MAP)

log_dir = f"runs/frozen-lake-td-" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S") + f"-{NUM_EPOCHS}-{NUM_STEPS}"
writer = SummaryWriter(log_dir)

pi = np.zeros(env.observation_space.n, dtype=np.int64)
Q = np.zeros((env.observation_space.n, env.action_space.n))
for epoch in tqdm.tqdm(range(NUM_EPOCHS)):
  alpha = 0.1
  Q, debug_info = eval_policy_td(env, MAP, pi, Q, 0.999, alpha, NUM_STEPS)
  pi = get_greedy_policy_from_Q(Q)

  if epoch % LOG_RATE == 0 or epoch == NUM_EPOCHS - 1:
    avg_return = policy_average_return(env, pi, GAMMA, 20)
    writer.add_scalar("Avg Return", avg_return, epoch)
    fig, ax = plt.subplots()
    h = visualize_policy(ax, MAP, np.max(Q, axis=-1), pi)
    fig.colorbar(h)
    writer.add_figure("Policy", fig, epoch)
    writer.flush()

writer.close()